# Example 11.4.3 / 1: Compare a Local and a Global Solver

Companion notebook for **11.4.3 Local and Global Nonlinear Solvers** in the AMPL lab Documentation, section "Compare a Local and a Global Solver". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compare a Local and a Global Solver

The objective $(x^2-1)^2+0.1x$ is nonconvex, with two wells. Pitfalls of Nonlinear Programming (AMPL lab documentation, section 11.4)
shows Ipopt landing in whichever well it starts in. A deterministic global solver on the same
bounded model returns the left-hand well from either start, because its answer rests on a bound
over the whole interval rather than on the start. The run records, for each solver and start, the
status, its numeric code `solve_result_num` (0 for Ipopt and 3 for Couenne, both in the
`solved` range), the point and the objective:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -2, <= 2;                      # finite bounds, needed by a global solver
minimize Wells: (x^2 - 1)^2 + 0.1 * x;  # nonconvex: two local minima

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for solver in ("ipopt", "couenne"):
    for start in (-1.5, 1.5):
        ampl.var["x"].set_value(start)
        ampl.solve(solver=solver, verbose=False)
        print(solver, start, ampl.solve_result, ampl.get_value("solve_result_num"),
              round(ampl.var["x"].value(), 6), round(ampl.obj["Wells"].value(), 6))

ipopt -1.5 solved 0 -1.012273 -0.100617
ipopt 1.5 solved 0 0.987257 0.099367
couenne -1.5 solved 3 -1.012273 -0.100617
couenne 1.5 solved 3 -1.012273 -0.100617


### Pure AMPL: model.run

Pure AMPL:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
for {s in {"ipopt", "couenne"}, start in {-1.5, 1.5}} {
    option solver (s);
    let x := start;
    solve > solve.log;
    printf "%s %g %s %d %.6f %.6f\n", s, start, solve_result, solve_result_num, x, Wells;
}

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
ipopt -1.5 solved 0 -1.012273 -0.100617
 
Ipopt 3.12.13: Optimal Solution Found
ipopt 1.5 solved 0 0.987257 0.099367

couenne: Optimal
couenne -1.5 solved 3 -1.012273 -0.100617

couenne: Optimal
couenne 1.5 solved 3 -1.012273 -0.100617
